# Design Spec — SQL-over-anything via DuckDB table functions
## Turn any REST / GraphQL / web API into a SQL-queryable table — **in-process**, as native DuckDB table functions

**Date:** 2026-05-31  •  **Status:** ✅ Approved — **Architecture Y (native DuckDB VTab)**  •  **Owner:** notebook team

> A user writes `SELECT * FROM polymarket_markets(active := true)` in a notebook cell; a Rust **adapter** registered as a DuckDB **table function** translates that into upstream API calls and returns Apache Arrow rows — **all inside the notebook's own DuckDB connection.** No separate server, no Arrow Flight, no airport extension. **Polymarket** is adapter #1.

> **Crate:** `spur-rest-table-gateway` at `crates/spur-notebook/rest-table-gateway/` — a *gateway* under which many REST/GraphQL API ↔ SQL table mappings are registered. (Renamed from the original `spur-flight-gateway`; there is no Flight under Architecture Y.)

This spec is **grounded** against the worktree via the `code_*` graph tools and against a **feasibility probe** (see the ADR). Integration touchpoints cite real `file:line` + stable symbol ids (see *Grounding Appendix*).

## 0. Architecture Decision Record — Flight/airport (X) → native DuckDB VTab (Y)

**Status:** Accepted 2026-05-31. Supersedes the original Arrow-Flight-server architecture; the X material is retained below (reframed) for rationale.

### Context
The first design (X) built a separate **Arrow Flight server** implementing the DuckDB **airport** extension's wire contract; the notebook's DuckDB would `ATTACH '…' (TYPE airport)` and query it. That works, but it carries the project's single biggest unknown — airport's **underdocumented MessagePack ticket + JSON predicate wire format** — plus a second process and a gRPC surface.

### Decision
Drop Flight/airport entirely. Register the REST adapters as **native DuckDB table functions** (duckdb-rs `vtab` feature) **directly on the notebook's existing DuckDB connection**. Same adapter engine, radically simpler front door.

### Evidence (feasibility probe, codex, `docs/spikes/duckdb-vtab-probe/`)
Verdict: **FEASIBLE-WITH-CAVEATS** against the pinned `duckdb 1.10502.0` (`bundled` + `vtab`).

| Probe question | Result |
|---|---|
| VTab API exists & links | ✅ `duckdb::vtab::VTab` + `Connection::register_table_function*` |
| Table function returns rows | ✅ `SELECT * FROM polymarket_markets('true')` read back |
| **Argument passing** (= filter → query param) | ✅ server saw `GET /markets?active=true` |
| async `reqwest` inside the sync VTab callback | ⚠️ **see caveat** |

**The caveat:** a VTab callback that calls `Runtime::block_on` **aborts the process** (`SIGABRT`, *"Cannot start a runtime from within a runtime"*) when the DuckDB query runs on a thread already inside a tokio runtime — i.e. the notebook's normal execution context. **Mitigation (all proven by the probe):** run DuckDB queries via `tokio::task::spawn_blocking` / a dedicated thread, and drive REST I/O on a **shared I/O runtime** the VTab callback waits on via a channel (never `block_on` on a worker thread). See §12.

### Consequences
- **Eliminates** the airport wire-format spike (Plan-1 task `t8` is now off-mission → recommend cancel).
- **Plan 1's adapter engine is unchanged and fully reused** — `Adapter::scan` feeds the VTab callback exactly as it would have fed a Flight `do_get`.
- **Loses** the X benefit "any external DuckDB client can ATTACH us" — Y is notebook-in-process only. Acceptable for a notebook-first product; X remains a possible future front door on the same engine.
- **UX shift:** DuckDB table functions are flat-named (`polymarket_markets`), not `schema.table`; filters arrive as **function arguments** first (true `WHERE`-pushdown is a later enhancement).

## 0a. ADR addendum — Y refined: loadable DuckDB extension into the Python kernel (post-spike, 2026-05-31)

**Status:** Accepted 2026-05-31. Refines §0's *"register on the notebook's DuckDB connection"* — that phrasing named the wrong connection.

### Correction
Plan-3 grounding showed the notebook executes cells in a **Python Jupyter kernel over ZeroMQ** (`jute-notebook/src-tauri/src/backend/local.rs`, `backend/wire_protocol/driver_zeromq.rs`); user SQL runs against the **Python `duckdb` package's connection** — a *separate OS process* from the Rust daemon. The Plan-2 VTab is registered on a **Rust** `duckdb::Connection`, so it is **invisible to the kernel**. Confirmed: `crates/spur-notebook/src/mcp/mod.rs:717-797` generates a Python `import duckdb; duckdb.sql("ATTACH …")` setup cell; the daemon holds no long-lived `Connection` (`datasource/mod.rs` only opens ephemeral in-memory connections for probing).

### Refined decision
Package `spur-rest-table-gateway` (the Plan-2 VTab + `IoBridge` + adapters) as a **loadable C-API DuckDB extension** (duckdb-rs `loadable-extension` + `#[duckdb_entrypoint_c_api]`). The managed setup cell **`LOAD`s** it into the kernel's duckdb — exactly where user SQL runs. Same engine, same `fn(active := true)` UX; only the **packaging + delivery** change.

### Evidence (spike, codex, `docs/spikes/duckdb-loadable-ext-probe/` — ran end-to-end)
| Question | Result |
|---|---|
| Rust-built **unsigned** `.duckdb_extension` LOADs in Python duckdb | ✅ `allow_unsigned_extensions=true` + local-path `LOAD` |
| Named-param table function via `:=` | ✅ `SELECT * FROM spur_probe(n := 5)` |
| Loads in the **actual SPUR kernel venv** | ✅ `~/.spur/jupyter/venv` already ships `duckdb 1.5.3` |
| Stable C API decouples from engine version | ✅ one artifact (C API `v1.2.0`) loaded in duckdb **1.5.2 and 1.5.3** |

### Caveats → Plan 3 must satisfy
- **Per-platform artifact** — build for the kernel's platform (e.g. `osx_arm64`); `scripts/spur-cargo` (remote linux) output will **not** load in a macOS kernel.
- **`allow_unsigned_extensions` is a connection-creation config** (cannot `SET` post-connect). The setup cell uses the **module-level default** `duckdb` connection, on which datasources become views; how to enable unsigned on the connection user SQL actually uses is **Plan 3's first (grounding) task**.
- **Pin `duckdb`** in the SPUR managed venv for a deterministic loader version; footer = platform + **C API** version; the duckdb-rs macro's `min_duckdb_version` is really the C-API version.
- **Reopen simplifies:** the regenerated setup cell re-`LOAD`s (Python-side) — no Rust connection-scoped re-registration.

Industry context: `docs/rca/2026-05-31-plan3-frontdoor-industry-crosscheck.md` (most tools ingest/materialize; the loadable extension is the path that preserves the live table-function UX).

## 1. Executive summary

- Expose Rust **REST adapters as DuckDB table functions** by shipping a **loadable C-API extension** the notebook's **Python kernel `LOAD`s** (duckdb-rs `vtab` + `loadable-extension`). The Plan-2 VTab runs *inside* the extension. No Flight, no airport, no second server. **(See ADR §0a — the kernel, not a Rust connection, is where it loads.)**
- Reuse the **Plan-1 adapter framework** (manifest → HTTP/pagination → JSON→Arrow → `Adapter` trait → registry → Polymarket adapter) verbatim — it is front-door-agnostic.
- **Adapter model is C3 (hybrid):** declarative **manifest** for clean REST tables + a Rust **`Adapter` trait** escape hatch for signing / GraphQL / cursor pagination / parameterized endpoints.
- **Async bridge (the one real constraint):** DuckDB queries run via `spawn_blocking`; a **shared I/O runtime** owns the `reqwest` client; the synchronous VTab callback sends a `ScanRequest` over a channel and blocks on the reply — never `block_on` on a tokio worker (which the probe proved aborts the process).
- **Integration:** a new `DatasourceKind::ApiTables` whose `add_api_datasource` emits a managed **setup cell that `LOAD`s the extension** (instead of `ATTACH (TYPE airport)`), wired through the existing `attach_datasource` / `reconcile_open_datasource_catalog` seam. Reopen re-runs the setup cell. (ADR §0a.)
- **Proof:** `SELECT question, volume FROM polymarket_markets(active := true)` and `SELECT * FROM polymarket_orderbook('0x…', depth := 50)`.

## 2. Goals / non-goals

### Goals
1. Query arbitrary REST/GraphQL APIs as SQL **table functions** from a notebook cell, no per-query glue.
2. Cheap to add a source: declarative manifest for the common case; a small trait impl for the hard case.
3. Push filters/args **down to the API** as query params when the API supports it.
4. Keep upstream API credentials **in-process and server-side** — never exposed to SQL output.
5. Stay **in the notebook process**, reusing its existing DuckDB connection and lifecycle.

### Non-goals (v1)
- **No Arrow Flight, no airport extension, no second process** (the whole point of Y).
- No external-client `ATTACH` reach (notebook-in-process only). X remains a possible future addition.
- No `WHERE`-clause filter pushdown in v1 — filters arrive as **function arguments**; `WHERE`-pushdown into table functions is a later enhancement.
- No writes (the table functions are read-only); no hot-reload of adapters; no multi-tenant credential isolation. (Deferred.)

## 3. Background — why in-process VTab beats Flight/airport here

The notebook **already embeds DuckDB** (`duckdb 1.10502.0`, `bundled`). The entire Arrow-Flight/airport machinery exists to connect a DuckDB client to a *remote, separate-process* data server. In our case the data "server" and the DuckDB client can be the **same process** — so that machinery is pure overhead.

| Concern | X — Flight + airport | **Y — native VTab (chosen)** |
|---|---|---|
| Biggest unknown | airport MessagePack ticket + JSON predicate wire format (reverse-engineering) | **resolved by probe** — duckdb-rs `vtab` API |
| Processes / DuckDBs | 2 / (1–2) | **1 / 1** |
| Filter → API param | designed, unproven | **proven** (function args) |
| async bridging | clean (Flight is async) | `spawn_blocking` + shared runtime (known, §12) |
| External clients ATTACH | yes | no (notebook-only) |
| Machinery | gRPC server + wire port | **register functions on a connection** |

Airport was the right tool *if* we needed a language/process boundary; we don't. Y trades that boundary for in-process simplicity and deletes the riskiest task in the project.

## 4. System context (everything in one process)

No gRPC, no Flight. The notebook's DuckDB calls Rust table functions; those reach the adapter engine through a shared I/O runtime; the engine calls REST APIs and returns Arrow rows.

```mermaid
flowchart LR
  subgraph User["Notebook UI (jute / Tauri)"]
    cell["SQL cell:\nSELECT * FROM polymarket_markets(active := true)"]
    panel["Datasource / functions panel"]
  end

  subgraph Proc["SPUR notebook process (Rust) — SINGLE process"]
    direction TB
    duck[("notebook DuckDB connection")]
    vtab["VTab table functions\n(registered on the connection)"]
    bridge["I/O bridge\nshared runtime + channel"]
    reg["AdapterRegistry"]
    adp["Adapters (manifest + trait)"]
  end

  subgraph World["World data APIs"]
    poly[["Polymarket REST"]]
    other[["any REST / GraphQL"]]
  end

  cell --> duck
  duck -- "bind / init / func\n(args = pushed filters)" --> vtab
  panel <-- "list registered functions" --> duck
  vtab -- "ScanRequest over channel\n(off the tokio worker)" --> bridge
  bridge --> reg --> adp
  adp -- "reqwest GET/POST + pagination + auth" --> poly
  adp --> other
  poly -- "JSON" --> adp
  adp -- "RecordBatch" --> bridge -- "rows" --> vtab -- "DataChunk" --> duck
```

## 5. Crate & module structure

The `airport/` module from the X design is replaced by a `vtab/` module. The `adapter/` framework (Plan 1) is unchanged. Crate: **`spur-rest-table-gateway`** at `crates/spur-notebook/rest-table-gateway/` (renamed from `spur-flight-gateway` — done).

```mermaid
flowchart TB
  subgraph crate["crates/spur-notebook/rest-table-gateway  (package spur-rest-table-gateway)"]
    lib["lib.rs\npublic API: register_source(conn, adapter)"]
    subgraph vtab["vtab/  — DuckDB integration (REPLACES airport/)"]
      reg2["register.rs\nConnection::register_table_function per table/TVF"]
      vt["table_fn.rs\nimpl VTab: bind / init / func\nargs -> Predicate ; RecordBatch -> DataChunk"]
      br["bridge.rs\nshared I/O runtime + channel\n(no block_on on a tokio worker)"]
    end
    subgraph adapter["adapter/  — framework (Plan 1, UNCHANGED)"]
      atr["mod.rs (Adapter trait + AdapterRegistry)"]
      man["manifest.rs"]
      mad["manifest_adapter.rs"]
      http["http.rs"]
      aj["json_to_batch.rs"]
    end
    subgraph adapters["adapters/"]
      pm["polymarket.rs"]
    end
    sec["secrets.rs"]
  end

  lib --> reg2 --> vt --> br --> atr
  atr --> man & http
  man --> mad --> http --> aj
  pm --> man
  pm -. "trait impl for orderbook TVF" .-> atr
  atr --> sec
```

## 6. The DuckDB VTab contract we implement (replaces the airport RPC table)

From `duckdb 1.10502.0` (confirmed by the probe):

```rust
pub trait VTab: Sized {
    type InitData: Sized + Send + Sync;
    type BindData: Sized + Send + Sync;
    fn bind(bind: &BindInfo) -> Result<Self::BindData, Box<dyn Error>>;   // declare result columns + read args
    fn init(init: &InitInfo) -> Result<Self::InitData, Box<dyn Error>>;   // per-scan state
    fn func(func: &TableFunctionInfo<Self>, output: &mut DataChunkHandle) -> Result<(), Box<dyn Error>>;  // emit rows
    fn parameters() -> Option<Vec<LogicalTypeHandle>>;                    // argument types
}
impl Connection {
    pub fn register_table_function<T: VTab>(&self, name: &str) -> Result<()>;
    pub fn register_table_function_with_extra_info<T: VTab, E: Clone + Send + Sync + 'static>(&self, name: &str, extra_info: &E) -> Result<()>;
}
```

| Step | What we do |
|---|---|
| `bind` | read function args via `BindInfo::get_parameter(i)`; map each to a `Predicate`; declare result columns via `add_result_column` (from the adapter's `TableDef.schema`); stash a `BindData { source, table, predicates }` |
| `init` | allocate per-scan cursor state |
| `func` | send `ScanRequest` to the I/O bridge, block on the reply, write the returned `RecordBatch` into the `DataChunkHandle` (`flat_vector` / `Inserter::insert` / `as_mut_slice`), `set_len` |
| register | `register_table_function_with_extra_info` once per table/TVF, passing the `AdapterRegistry` handle as `extra_info` |

**Skipped (v1):** projection-pushdown via `init` (nice-to-have), `WHERE`-filter pushdown, writes.

## 7. Query lifecycle (sequence)

End-to-end for `SELECT question, volume FROM polymarket_markets(active := true)`. Note the query runs via `spawn_blocking` so the VTab callback is **off** the tokio worker.

```mermaid
sequenceDiagram
  participant Cell as Notebook SQL cell
  participant Duck as notebook DuckDB
  participant VT as VTab (bind/init/func)
  participant Br as I/O bridge (shared runtime)
  participant Reg as AdapterRegistry
  participant Adp as Adapter (polymarket)
  participant API as Polymarket REST

  Cell->>Duck: SELECT ... FROM polymarket_markets(active := true)
  Note over Duck: query executed via spawn_blocking (off tokio worker)
  Duck->>VT: bind() -> read args, declare columns
  VT->>VT: arg active=true -> Predicate{active = true}
  Duck->>VT: init()
  Duck->>VT: func(output chunk)
  VT->>Br: send ScanRequest{table:"markets", predicates, projection}
  Br->>Reg: route source "polymarket"
  Reg->>Adp: scan(req)
  Adp->>API: GET /markets?active=true&limit=500&offset=N (paginated)
  API-->>Adp: JSON pages
  Adp-->>Br: Vec<RecordBatch>
  Br-->>VT: rows (blocking recv on channel)
  VT->>Duck: write rows into DataChunk; set_len
  Duck-->>Cell: result rows
```

## 8. Filter & TVF argument flow

Under Y, both clean tables and parameterized endpoints arrive the **same way** — as DuckDB **table-function arguments** — and converge on `Adapter::scan`. Named args map to manifest `[table.filters]` (→ query param) or to a TVF's positional params. Args the adapter can't push are simply applied to the request it builds; any leftover row filtering DuckDB still does in the outer query.

```mermaid
flowchart TB
  start(["VTab bind(): function args"]) --> kind{"manifest table\nor TVF?"}

  kind -- "manifest table" --> nm["named arg active := true"]
  nm --> map{"[table.filters] maps arg -> param?"}
  map -- yes --> q1["push down: ?active=true"]
  map -- no --> resid["ignored at API; DuckDB filters rows in outer query"]

  kind -- "TVF" --> pos["positional args: orderbook('0x..', depth := 50)"]
  pos --> tvf["adapter maps args -> path/query"]

  q1 --> req["build ScanRequest{predicates}"]
  resid --> req
  tvf --> req
  req --> bridge["I/O bridge -> Adapter::scan -> http.rs (paginate, auth, cache)"]
  bridge --> conv["json_to_batch: declared-schema typed RecordBatch"]
  conv --> chunk(["func writes DataChunk"])
```

## 9. Adapter model (C3) — manifest + trait (unchanged from Plan 1, front-door-agnostic)

The adapter layer does not know whether it is driven by a Flight `do_get` or a DuckDB VTab `func` — it just answers a `ScanRequest` with Arrow. This is exactly why the Plan-1 work survived the X→Y pivot untouched.

### 9a. Declarative manifest (clean REST tables)
```toml
[source]
name = "polymarket"
base_url = "https://gamma-api.polymarket.com"
auth = { scheme = "none" }
pagination = { style = "offset", limit_param = "limit", offset_param = "offset", page_size = 500 }
[[table]]
name = "markets"
path = "/markets"
[table.columns]
id       = { json = "$.id",       type = "Utf8" }
question = { json = "$.question", type = "Utf8" }
active   = { json = "$.active",   type = "Boolean" }
volume   = { json = "$.volume",   type = "Float64" }
[table.filters]
active = { param = "active" }
```
Each `[[table]]` becomes one registered table function `polymarket_markets(...)`.

### 9b. Code escape hatch (the trait)
```rust
#[async_trait]
pub trait Adapter: Send + Sync {
    fn name(&self) -> &str;
    fn catalog(&self) -> Vec<TableDef>;   // tables + TVFs, each with a declared Arrow schema
    async fn scan(&self, req: ScanRequest) -> Result<Vec<RecordBatch>>;
}
```
The framework owns HTTP, pagination, rate-limiting, caching, JSON→Arrow. An adapter author describes endpoints (manifest) and optionally shapes requests/responses (trait).

```mermaid
classDiagram
  class Adapter {
    <<trait>>
    +name() str
    +catalog() Vec~TableDef~
    +scan(ScanRequest) Vec~RecordBatch~
  }
  class AdapterRegistry { +register(Adapter) +get(source) Adapter +sources() }
  class ManifestAdapter { -manifest +catalog() +scan() }
  class PolymarketAdapter { -inner: ManifestAdapter +catalog() +scan() }
  class VTabTableFn { +bind() +init() +func() }
  class IoBridge { +call(ScanRequest) Vec~RecordBatch~ }
  Adapter <|.. ManifestAdapter
  Adapter <|.. PolymarketAdapter
  PolymarketAdapter o-- ManifestAdapter
  AdapterRegistry o-- Adapter
  VTabTableFn --> IoBridge
  IoBridge --> AdapterRegistry
```

## 10. The async bridge (the one real constraint — from the probe)

The adapters are **async** (`reqwest`); a VTab `func` is **synchronous** and is invoked on whatever thread runs the query. The probe proved that calling `Runtime::block_on` inside `func` **aborts the process** (`SIGABRT`) when that thread is already inside a tokio runtime — which is the notebook's normal state. The bridge exists to make this impossible by construction.

```mermaid
flowchart TB
  q["DuckDB query containing a REST table function"] --> sb["run via tokio::task::spawn_blocking\n(or dedicated query thread)"]
  sb --> func["VTab func() — synchronous, OFF the tokio worker"]
  func --> chan["send ScanRequest on a channel"]
  chan --> iort["shared I/O runtime thread\nowns reqwest client, drives async scan"]
  iort --> reply["Vec<RecordBatch> back on the channel"]
  reply --> emit["func writes DataChunk"]

  bad["ANTIPATTERN: func calls Runtime::block_on\nwhile on a tokio worker thread"] --> abort["SIGABRT: cannot start a runtime within a runtime\n(proven by probe — uncatchable)"]
```

**Rules baked into `bridge.rs`:** (1) one shared I/O runtime for the whole gateway, created once; (2) `func` only ever does a blocking channel `recv`, never `block_on`; (3) the notebook backend must execute DuckDB queries that touch these functions via `spawn_blocking`.

## 11. Notebook integration (grounded in current `spur-notebook` code)

Under Y the integration is **simpler** than X — there is no gateway process to lifecycle-manage and no `ATTACH`. Enabling a source = **registering its table functions** on the notebook's DuckDB connection. Each box cites a real symbol (see *Grounding Appendix*).

### The seam
The existing path-based attach (`attach_datasource`, `crates/spur-notebook/src/mcp/mod.rs:1085`) infers a kind from a file path and calls `introspect_datasource` (`datasource/mod.rs:24`). An API source has **no path**; instead it carries a **source name + adapter manifest**, and its "attach" registers functions. We add a parallel route rather than overloading path inference.

```mermaid
flowchart TB
  subgraph existing["EXISTING — path-based datasources (unchanged)"]
    a1["attach_datasource(name, path, group)\nmcp/mod.rs:1085"]
    a3["infer_datasource_kind(&path)"]
    a4["introspect_datasource(&path, kind)\ndatasource/mod.rs:24"]
    a1 --> a3 --> a4
  end

  subgraph new["NEW — API table-function source (Y)"]
    b1["add_api_datasource(name, source/manifest)\n(new daemon method)"]
    b2["load adapter into AdapterRegistry"]
    b3["register_table_function per table/TVF\non the notebook DuckDB conn (spawn_blocking)"]
    b4["catalog from Adapter::catalog() — no network probe needed"]
    b1 --> b2 --> b3 --> b4
  end

  subgraph shared["SHARED catalog surface"]
    c1["DatasourceEntry { name, kind=ApiTables, group, columns/tables }\ncommands.rs:163 + acp events.rs:124"]
    c2["refresh_datasource_setup_cell()"]
    c4["reconcile_open_datasource_catalog() mcp/mod.rs:1611\n-> re-register functions on reopen"]
    c5["list_datasources tools/mod.rs:19 -> UI panel"]
  end

  a4 --> c1
  b4 --> c1
  c1 --> c2
  c4 -. "on reopen: re-register (functions are connection-scoped)" .-> b3
  c1 --> c5
```

### 11b. Register & re-register sequence

DuckDB table functions are **connection-scoped** — a fresh connection (notebook reopen / kernel restart) needs them re-registered. The existing `reconcile_open_datasource_catalog` (`mcp/mod.rs:1611`) reopen hook is the natural place.

```mermaid
sequenceDiagram
  participant UI as Notebook UI
  participant D as NotebookDaemonControl
  participant GW as gateway (registry + bridge)
  participant Duck as notebook DuckDB

  Note over UI,Duck: Enable a source
  UI->>D: add_api_datasource(name, manifest)
  D->>GW: load adapter into AdapterRegistry
  D->>Duck: spawn_blocking: register_table_function per table/TVF
  D->>D: DatasourceEntry{kind: ApiTables} + refresh setup cell

  Note over UI,Duck: Reopen notebook / restart kernel
  UI->>D: notebook open
  D->>D: reconcile_open_datasource_catalog() (mod.rs:1611)
  alt entry.kind == ApiTables
    D->>GW: ensure adapter loaded
    D->>Duck: re-register functions on the fresh connection
  end
```

## 12. Type & contract changes

| Change | File (grounded, current line) | Note |
|---|---|---|
| Add `ApiTables` variant to `DatasourceKind` | `jute-notebook/src-tauri/src/commands.rs:123` (canonical) + `crates/spur-acp/src/domain/events.rs:96` (ACP mirror) | `#[ts]` regenerates `DatasourceKind.ts` (→ `"api_tables"`); add match arms in `tests/datasource_wire_contract.rs` |
| Datasource locator for an API source | `DatasourceEntry` `commands.rs:165` + `events.rs:124` | `path` is currently required `String`; store the extension/source locator there, or add an optional `source` field (update **both** structs + the wire-contract test) |
| **Package gateway as a loadable extension** | new build target in `crates/spur-notebook/rest-table-gateway` (`loadable-extension` + `#[duckdb_entrypoint_c_api]`) | per-platform `.duckdb_extension`; footer = platform + **C API** version (`v1.2.0`); Plan-2 VTab/`IoBridge`/adapters compiled in |
| **Pin `duckdb` in the managed kernel venv** | `jute-notebook/src-tauri/src/kernel_provision.rs:129-190` | deterministic loader version (venv already ships `duckdb 1.5.3`) |
| New daemon route `add_api_datasource` | `crates/spur-notebook/src/mcp/mod.rs` near `:1090` | records the entry + refreshes setup cell; catalog from `Adapter::catalog()` (no network probe) |
| **Setup cell `LOAD`s the extension (unsigned)** | `datasource_setup_*` `mcp/mod.rs:717-797` | connection must be created with `allow_unsigned_extensions`; **resolve the default-connection model (Plan 3 task 1)** |
| Re-`LOAD` on reopen | regenerated setup cell (Python) | functions live in the **kernel** — no Rust re-registration (supersedes §11b's "connection-scoped" framing) |

> **Wire-compat caution:** `DatasourceKind` / `DatasourceEntry` are duplicated across `jute::commands` and `spur-acp::domain::events` and exported to TS; change all three together (the `datasource_wire_contract` test guards this).

## 13. Auth & secrets

No network trust boundary to cross anymore (it's all one process). Upstream credentials are resolved **in-process** in `secrets.rs`, attached per-source in `Adapter::scan`, and **never written into result columns** returned to SQL.

```mermaid
flowchart LR
  vt["VTab func"] --> br["I/O bridge"] --> sec["secrets.rs resolve per-source creds"]
  sec -- "env / secret store" --> store[("credential source")]
  sec -- "ResolvedAuth (stays in process)" --> adp["Adapter::scan"]
  adp -- "authd request" --> api[["upstream API"]]
  api -. "API key never enters a result column" .-x vt
```

## 14. Error handling

- **Adapter / HTTP errors** → returned from `func` as `Err(Box<dyn Error>)` → surfaced as a clean DuckDB error in the cell.
- **Schema drift** → declared-schema typed coercion with a precise `field X expected T` error; never silent null-fill.
- **Rate limits / HTTP 429** → retry-with-backoff in `http.rs`; not cached.
- **Nested-runtime misuse** → structurally prevented: `func` never calls `block_on`; the backend runs touching queries via `spawn_blocking` (§10). A debug-assert can detect "am I on a tokio worker?" to fail loudly in tests.
- **Bridge timeout** → if the I/O runtime doesn't reply within a bound, `func` returns an error rather than hanging the query thread.

## 15. Testing strategy

| Layer | What | How |
|---|---|---|
| **Adapter contract** | pagination, arg→param mapping, schema coercion, TVF args | `wiremock` fake REST server — offline, deterministic (Plan 1) |
| **JSON→Arrow** | typed coercion, drift errors | unit tests on `json_to_batch.rs` |
| **VTab registration** | function registers; `SELECT * FROM fn(...)` returns rows; arg reaches the request | real `duckdb` + `vtab`, wiremock upstream (mirrors the probe) |
| **Async bridge** | query inside an outer tokio runtime via `spawn_blocking` does **not** abort; direct `block_on` path is rejected | the probe's exact scenarios, promoted to tests |
| **Live Polymarket** | smoke test | `#[ignore]` / manual — never in CI |

The probe at `docs/spikes/duckdb-vtab-probe/` is the template for the VTab + bridge tests.

## 16. Proof — Polymarket as adapter #1

**Manifest table (named-arg filter):**
```sql
SELECT question, volume
FROM polymarket_markets(active := true)   -- active -> ?active=true
ORDER BY volume DESC
LIMIT 10;
```

**TVF via trait escape hatch:**
```sql
SELECT price, size
FROM polymarket_orderbook('0xabc...token', depth := 50);  -- args -> path/query
```

Polymarket validates both halves of C3: the manifest yields `polymarket_markets` / `polymarket_events`; a ~20-line trait impl yields the parameterized `polymarket_orderbook` TVF. (Note the flat function names — DuckDB VTabs are not `schema.table`.)

## 17. Phasing / plans

```mermaid
flowchart LR
  P1["Plan 1 — adapter core (RUNNING)\nmanifest, http, json->arrow, Adapter+registry, Polymarket\n(front-door-agnostic; unaffected by X->Y)"]
  P2["Plan 2 — VTab integration (Y)\nvtab/: register.rs, table_fn.rs (bind/init/func),\nbridge.rs (shared runtime + spawn_blocking), args->Predicate"]
  P3["Plan 3 — notebook wiring (EXTENSION front door)\nDatasourceKind::ApiTables, loadable-ext packaging + venv duckdb pin,\nadd_api_datasource -> setup-cell LOAD (unsigned), functions panel"]
  P1 --> P2 --> P3
```

- **Plan 1** keeps running unchanged — its output is the engine both X and Y consume.
- **Plan 2** is the new airport-free front door: the `vtab/` module + the async bridge. The former `t8` airport wire-format spike is **cancelled** (off-mission under Y).
- **Plan 3** wires it in via a **loadable extension the kernel `LOAD`s** (ADR §0a; spike `docs/spikes/duckdb-loadable-ext-probe` = FEASIBLE): kind + extension packaging/per-platform build + venv `duckdb` pin + `add_api_datasource`(setup-cell `LOAD`) + reopen re-`LOAD` + functions panel. Task 1 grounds the `allow_unsigned` connection model.

## 18. Decisions

1. **Front door:** ✅ **Y, refined → a loadable C-API DuckDB extension the Python kernel `LOAD`s** (ADR §0a). The Plan-2 VTab runs inside it. (Earlier *"register on the notebook connection"* corrected: notebook SQL runs in the Python kernel — a separate process.)
2. **Crate name:** ✅ **Done** — renamed `spur-flight-gateway` → **`spur-rest-table-gateway`** (dir `crates/spur-notebook/rest-table-gateway/`). A *gateway* under which many REST↔table mappings register.
3. **Function naming:** flat `polymarket_markets`, `polymarket_orderbook` (DuckDB VTabs aren't `schema.table`).
4. **Filters:** ✅ **function arguments** in v1 (`fn(active := true)`); `WHERE`-pushdown into table functions is a later enhancement.
5. **Async bridge:** ✅ shared I/O runtime + `spawn_blocking`; `func` never `block_on`s (probe-mandated).
6. **`t8` airport spike:** ✅ rejected (off-mission), but its wire-format reference docs were preserved to `docs/architecture/airport-wire-format.md` + `airport-fixtures/` for a possible *future* Architecture X (external `ATTACH`).
7. **Datasource locator:** ✅ additive optional `source` field; keep `path` optional (low wire-contract churn).
8. **Plan 1 status:** ✅ **merged to `main` + 8/8 tests green** (commits `0c598bca…8ff125a1`, renamed in `45b8fc2c`).
9. **Plan 2 status:** ✅ **merged to `main`** (VTab + `IoBridge` + grounding fixes; e2e green). Validated as an *engine*; its front door is now the extension (ADR §0a).
10. **Kernel reality:** ✅ notebook SQL executes in a **Python Jupyter kernel over ZeroMQ**; the SPUR managed venv already ships `duckdb 1.5.3`. Plan 3 pins it.
11. **Extension feasibility:** ✅ spike `docs/spikes/duckdb-loadable-ext-probe` — unsigned local-path Rust extension loads in the real kernel; named-param table fns work; stable **C API `v1.2.0`** loads across duckdb 1.5.2/1.5.3.
12. **Open (Plan 3 task 1):** how to enable `allow_unsigned_extensions` on the connection user SQL uses (module-level default vs a shared configured connection).

## 19. Grounding appendix

**Code graph** (hash `f76ef47d…`, head `8a6522e0…`, `response_file_oids_match: true`):

| Symbol | Kind | file:line | stable id |
|---|---|---|---|
| `DatasourceKind` (canonical) | enum | `crates/spur-notebook/jute-notebook/src-tauri/src/commands.rs:121` | `fc0a4090d5e2c116` |
| `DatasourceKind` (ACP mirror) | enum | `crates/spur-acp/src/domain/events.rs:96` | `4d960237dd35e917` |
| `DatasourceEntry` (canonical) | struct | `commands.rs:163` | `1e8e7e06ee714fe5` |
| `introspect_datasource` | fn | `crates/spur-notebook/src/datasource/mod.rs:24` | `2fc20d846ceee96f` |
| `attach_datasource` | method | `crates/spur-notebook/src/mcp/mod.rs:1085` | `7ba96d0162084f4e` |
| `reconcile_open_datasource_catalog` | method | `mcp/mod.rs:1611` | `ded9a6e26533f1db` |
| `list_datasources` | module | `crates/spur-notebook/src/mcp/tools/mod.rs:19` | `239ea0af390fa441` |

**Feasibility probe** (Architecture Y): `docs/spikes/duckdb-vtab-probe/` — verdict **FEASIBLE-WITH-CAVEATS**; duckdb-rs `vtab` API confirmed; nested-runtime abort + `spawn_blocking` mitigation proven. (On worker branch `spur/worker/v2/codex/…/0da3be96…` until landed.)

**External references**
- duckdb-rs `vtab`: <https://docs.rs/duckdb/latest/duckdb/vtab/>
- (superseded front door) airport extension: <https://airport.query.farm/> · quackflight reference: <https://github.com/quackscience/quackflight>